In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from OSDE.LegendreExpSPDensity import LegExp, LegExpSPDensity, LegExpResult
from StocProcess.RBM import RBMTransProb, MakeRBMTransProbFunc
from QAE.RQAE import RQAE

In [2]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
mu = 0.5
sigma = 1.0
n_terms = 5

# approximation setting
maxDeg = 5
R = 12
eps0 = 1 / 2**6

In [3]:
Ns = (2 ** np.linspace(3, 7, 9)).astype(int)
print(Ns)

[  8  11  16  22  32  45  64  90 128]


In [ ]:
epss = []
pEsts = []
totalQueryNums = []
maxDepths = []

for N in Ns:
    legExpResultPrev = None
    ts = np.concatenate([[0], np.linspace(0.2, 0.6, N)])
    eps = eps0 / np.sqrt(N)
    epss.append(eps)
    totalQueryNum = 0
    maxDepth = 0
    print(datetime.datetime.now(), "N=", N)

    for i in range(len(ts)-1):
        print("i_t=", i, datetime.datetime.now())
        transProbFunc = MakeRBMTransProbFunc(ts[i+1], ts[i], c, d, mu, sigma, n_terms)

        if i == 0:
            densFunc = lambda x: transProbFunc(x, x0)
            legExpResult = LegExp(densFunc, maxDeg)
        else:
            legExpResult = LegExpSPDensity(legExpResultPrev.fApp, transProbFunc, maxDeg)

        coefs = np.zeros(maxDeg+1)
        coefs[0] = 0.5

        for l in range(1, maxDeg+1):
            a = 0.5 * (legExpResult.coefs[l] / (l + 0.5) + 1)
            rqaeResult = RQAE(a, eps, R)
            coefs[l] = (2 * rqaeResult.aEst -1) * (l + 0.5)
            totalQueryNum += rqaeResult.TotalQueryNum
            maxDepth = max(maxDepth, rqaeResult.MaxDepth)

        legExpResultPrev = LegExpResult(coefs)

    pEsts.append(sp.integrate.quad(legExpResultPrev.fApp, x0, 1.0)[0])
    totalQueryNums.append(totalQueryNum)
    maxDepths.append(maxDepth)

2025-01-28 17:54:29.170888 N= 8
i_t= 0 2025-01-28 17:54:29.170888


c:\Users\koich\Desktop\Code\DivQCOSDE\QAE\MaximizeL.py:11: RuntimeWarning: divide by zero encountered in log
  neglogL = lambda theta: -np.dot(n1s, np.log(np.sin(thetaMuls * theta)**2)) - np.dot(n0s, np.log(np.cos(thetaMuls * theta)**2))


i_t= 1 2025-01-28 17:54:30.328354
i_t= 2 2025-01-28 17:54:57.882495
i_t= 3 2025-01-28 17:55:23.348924
i_t= 4 2025-01-28 17:55:40.289533
i_t= 5 2025-01-28 17:55:50.256659
i_t= 6 2025-01-28 17:56:11.009510
i_t= 7 2025-01-28 17:56:32.518996
2025-01-28 17:56:54.042474 N= 11
i_t= 0 2025-01-28 17:56:54.042474
i_t= 1 2025-01-28 17:56:55.356407
i_t= 2 2025-01-28 17:57:26.021024
i_t= 3 2025-01-28 17:58:01.245207
i_t= 4 2025-01-28 17:58:31.074501
i_t= 5 2025-01-28 17:58:50.076983
i_t= 6 2025-01-28 17:59:11.450916
i_t= 7 2025-01-28 17:59:41.241729
i_t= 8 2025-01-28 18:00:12.018927
i_t= 9 2025-01-28 18:00:37.110742
i_t= 10 2025-01-28 18:00:59.961846
2025-01-28 18:01:24.023456 N= 16
i_t= 0 2025-01-28 18:01:24.023456
i_t= 1 2025-01-28 18:01:25.640779
i_t= 2 2025-01-28 18:02:16.807095
i_t= 3 2025-01-28 18:03:04.851633
i_t= 4 2025-01-28 18:03:55.330842
i_t= 5 2025-01-28 18:04:42.940495
i_t= 6 2025-01-28 18:05:30.277602
i_t= 7 2025-01-28 18:06:20.092408
i_t= 8 2025-01-28 18:07:09.252640
i_t= 9 2025-01-

In [5]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(ts[-1], t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [6]:
retDf = pd.DataFrame(dict(N=Ns,
                          pTrue=np.repeat(pTrue, len(Ns)),
                          eps=epss,
                          pEst=pEsts,
                          absErr=np.abs(np.array(pEsts) - pTrue),
                          totalQueryNum=totalQueryNums,
                          maxDepth=maxDepths))

In [7]:
retDf.to_csv('DivideRBM_RQAE.csv', index=False)

In [8]:
retDf

,N,pTrue,eps,pEst,absErr,totalQueryNum,maxDepth
0,8,0.649605,0.005524,0.649062,0.000543,163720,181
1,11,0.649605,0.004711,0.645918,0.003687,235634,212
2,16,0.649605,0.003906,0.652256,0.002651,362301,255
3,22,0.649605,0.003331,0.647287,0.002318,865643,300
4,32,0.649605,0.002762,0.649156,0.000449,1319916,362
5,45,0.649605,0.002329,0.654919,0.005314,1949924,429
6,64,0.649605,0.001953,0.651550,0.001945,2929294,511
7,90,0.649605,0.001647,0.650637,0.001032,7141456,607
8,128,0.649605,0.001381,0.642774,0.006831,10596302,724
